In [2]:
"""Module for querying indexed LinkedIn profile data."""

import logging
from typing import Any, Dict, Optional
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

from llama_index.core import VectorStoreIndex, PromptTemplate


logger = logging.getLogger(__name__)

LLM_MODEL_ID = "ibm/granite-3-2-8b-instruct"
EMBEDDING_MODEL_ID = "llama3"
SIMILARITY_TOP_K = 5
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 500
MIN_NEW_TOKENS = 1
TOP_K = 50
TOP_P = 1
INITIAL_FACTS_TEMPLATE = """
You are an AI assistant that provides detailed answers based on the provided context.

Context information is below:

{context_str}

Based on the context provided, list 3 interesting facts about this person's career or education.

Answer in detail, using only the information provided in the context.
"""

USER_QUESTION_TEMPLATE = """
You are an AI assistant that provides detailed answers to questions based on the provided context.

Context information is below:

{context_str}

Question: {query_str}

Answer in full details, using only the information provided in the context. If the answer is not available in the context, say "I don't know. The information is not available on the LinkedIn page."
"""

def create_llm(
    temperature: float = TEMPERATURE,
    max_new_tokens: int = MAX_NEW_TOKENS,
    decoding_method: str = "sample"
) -> WatsonxLLM:
    """Creates an IBM Watsonx LLM for generating responses.
    
    Args:
        temperature: Temperature for controlling randomness in generation (0.0 to 1.0).
        max_new_tokens: Maximum number of new tokens to generate.
        decoding_method: Decoding method to use (sample, greedy).
        
    Returns:
        llama model.
    """
    additional_params = {
        "decoding_method": decoding_method,
        "min_new_tokens": config.MIN_NEW_TOKENS,
        "top_k": config.TOP_K,
        "top_p": config.TOP_P,
    }
    
    llm = ChatOllama(model='llama3')
    chain = promptZeroShot | llm | StrOutputParser 
    responseChain = chain.invoke({'concept':'engrais'})
    logger.info(f"Created llama LLM model: {LLM_MODEL_ID}")
    
    return llm
    
def generate_initial_facts(index: VectorStoreIndex) -> str:
    """Generates interesting facts about the person's career or education.
    
    Args:
        index: VectorStoreIndex containing the LinkedIn profile data.
        
    Returns:
        String containing interesting facts about the person.
    """
    try:
        # Create LLM for generating facts
        LLAMA_LLM = create_llm(
            temperature=0.0,
            max_new_tokens=500,
            decoding_method="sample"
        )
        
        # Create prompt template
        INITIAL_PROMPT = PromptTemplate(template=INITIAL_FACTS_TEMPLATE)
        
        chain = INITIAL_PROMPT | LLAMA_LLM | JsonOutputParser()

        INITIAL_QUERY = 'agriculure'
        result = chain.invoke({"context_str": INITIAL_QUERY})
        return result
    except Exception as e:
        logger.error(f"Error in generate_initial_facts: {e}")
        return "Failed to generate initial facts."

def answer_user_query(index: VectorStoreIndex, user_query: str) -> Any:
    """Answers the user's question using the vector database and the LLM.
    
    Args:
        index: VectorStoreIndex containing the LinkedIn profile data.
        user_query: The user's question.
        
    Returns:
        Response object containing the answer to the user's question.
    """
    try:
        # Create LLM for answering questions
        LLAMA_LLM = create_llm(
            temperature=0.0,
            max_new_tokens=500,
            decoding_method="sample"
        )
        
        # Create prompt template
        user_prompt = PromptTemplate(template=USER_QUESTION_TEMPLATE)
        
        chain = user_prompt | LLAMA_LLM | JsonOutputParser()

        answer = chain.invoke({"context_str": user_query})
        return answer
    except Exception as e:
        logger.error(f"Error in answer_user_query: {e}")
        return "Failed to get an answer."

NameError: name 'WatsonxLLM' is not defined